# Outline

- Perform Preprocessing
- Choose a model
- Make a pickle file for Pipeline and Model
- Test it

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
X_train = pd.read_csv('../Feature Selection/X_train_houses.csv',index_col=0,)
X_test = pd.read_csv('../Feature Selection/X_test_houses.csv',index_col=0)
y_train = pd.read_csv('../Feature Selection/y_train_houses.csv',index_col=0)
y_test = pd.read_csv('../Feature Selection/y_test_houses.csv',index_col=0)

In [4]:
X_train.head()

,Property era_Brand New (2025),Property era_Recently Built,Property era_Vintage,Main Location,Area(Marla),Bath(s),Bedroom(s),Kitchens,Storey Unit,IsPrimeLoc,SolarInstalled,WaterBore,CornerHouse
906,1.0,0.0,0.0,2.384121,2.079442,5.0,5.0,2.0,2.0,False,False,False,False
3379,1.0,0.0,0.0,1.555500,2.397895,6.0,5.0,2.0,2.0,False,False,False,False
2094,1.0,0.0,0.0,1.496598,1.791759,4.0,3.0,2.0,2.0,True,False,False,True
218,1.0,0.0,0.0,2.258986,2.708050,6.0,6.0,1.0,2.0,True,False,False,False
3455,1.0,0.0,0.0,2.012407,1.609438,5.0,5.0,2.0,2.0,True,False,False,False


In [5]:
X_train.info()

<class 'pandas.DataFrame'>
Index: 2878 entries, 906 to 1246
Data columns (total 13 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Property era_Brand New (2025)  2878 non-null   float64
 1   Property era_Recently Built    2878 non-null   float64
 2   Property era_Vintage           2878 non-null   float64
 3   Main Location                  2878 non-null   float64
 4   Area(Marla)                    2878 non-null   float64
 5   Bath(s)                        2878 non-null   float64
 6   Bedroom(s)                     2878 non-null   float64
 7   Kitchens                       2878 non-null   float64
 8   Storey Unit                    2878 non-null   float64
 9   IsPrimeLoc                     2878 non-null   bool   
 10  SolarInstalled                 2878 non-null   bool   
 11  WaterBore                      2878 non-null   bool   
 12  CornerHouse                    2878 non-null   bool   
dtypes:

In [6]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score,mean_absolute_error

In [7]:
y_train_log = np.log1p(y_train)

With y_train log transformed

In [8]:
clf = RandomForestRegressor(random_state=42)
clf.fit(X_train,y_train_log)
y_pred = clf.predict(X_test)

C:\Users\Wajih\anaconda3\Lib\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


In [9]:
y_pred = np.expm1(y_pred)
print('r2 Score',r2_score(y_test,y_pred))
print('Mean absolute error',mean_absolute_error(y_test,y_pred))

r2 Score 0.9357857817229983
Mean absolute error 1.2948798979398521


Without y_train log transformed

In [12]:
clf = RandomForestRegressor(random_state=42)
clf.fit(X_train,y_train)
y_pred = clf.predict(X_test)

C:\Users\Wajih\anaconda3\Lib\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


In [13]:
# y_pred = np.expm1(y_pred)
print('r2 Score',r2_score(y_test,y_pred))
print('Mean absolute error',mean_absolute_error(y_test,y_pred))

r2 Score 0.9329584841113832
Mean absolute error 1.3025587704036652


With y_train as log_transformed, it is giving good results

In [14]:
cols_to_scale = ['Main Location','Area(Marla)', 'Bath(s)', 'Bedroom(s)', 'Kitchens']

In [15]:
X_train[cols_to_scale].describe()

,Main Location,Area(Marla),Bath(s),Bedroom(s),Kitchens
count,2878.000000,2878.000000,2878.000000,2878.000000,2878.000000
mean,2.004029,2.436152,5.362057,5.023280,1.815497
std,0.687974,0.578076,1.299546,1.504187,0.487247
min,0.729172,1.098612,1.000000,1.000000,1.000000
25%,1.496598,1.945910,5.000000,4.000000,2.000000
50%,1.811100,2.397895,6.000000,5.000000,2.000000
75%,2.384121,3.044522,6.000000,6.000000,2.000000
max,4.168861,4.204693,16.000000,14.000000,3.000000


In [18]:
from sklearn.preprocessing import RobustScaler
from sklearn.compose import ColumnTransformer

from sklearn import set_config

set_config(transform_output='pandas')

In [19]:
scalar = ColumnTransformer(
    [
        ('Standard Scalar',RobustScaler(),cols_to_scale)
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
)

In [20]:
X_train_transformed = scalar.fit_transform(X_train)
X_test_transformed = scalar.transform(X_test)

In [21]:
clf = RandomForestRegressor(random_state=42)
clf.fit(X_train_transformed,y_train_log)
y_pred = clf.predict(X_test_transformed)

C:\Users\Wajih\anaconda3\Lib\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


In [22]:
y_pred = np.expm1(y_pred)
print('r2 Score',r2_score(y_test,y_pred))
print('Mean absolute error',mean_absolute_error(y_test,y_pred))

r2 Score 0.9368592045039722
Mean absolute error 1.2926008812645144


In [23]:
from xgboost import XGBRegressor

In [24]:
clf = XGBRegressor()
clf.fit(X_train_transformed,y_train_log)
y_pred = clf.predict(X_test_transformed)

In [25]:
y_pred = np.expm1(y_pred)
print('r2 Score',r2_score(y_test,y_pred))
print('Mean absolute error',mean_absolute_error(y_test,y_pred))

r2 Score 0.9232802987098694
Mean absolute error 1.32845139503479


In [26]:
from sklearn.ensemble import ExtraTreesRegressor

In [27]:
clf = ExtraTreesRegressor()
clf.fit(X_train_transformed,y_train_log)
y_pred = clf.predict(X_test_transformed)

C:\Users\Wajih\anaconda3\Lib\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


In [28]:
y_pred = np.expm1(y_pred)
print('r2 Score',r2_score(y_test,y_pred))
print('Mean absolute error',mean_absolute_error(y_test,y_pred))

r2 Score 0.9321822662412108
Mean absolute error 1.3448687778904485


In [29]:
y_pred_train_log = clf.predict(X_train_transformed)
y_pred_train_real = np.expm1(y_pred_train_log)
print('r2 Score',r2_score(y_train,y_pred_train_real))
print('Mean absolute error',mean_absolute_error(y_train,y_pred_train_real))

r2 Score 0.9979746153509785
Mean absolute error 0.15333750594648263


## Optuna (for Hyperparameter Tunning)

In [30]:
from sklearn.model_selection import StratifiedKFold

In [31]:
import optuna
from optuna.visualization import plot_contour,plot_optimization_history,plot_parallel_coordinate,plot_param_importances

In [32]:
y_train_log

,Price(Cr)
906,2.374906
3379,1.704748
2094,1.280934
218,2.287471
3455,1.504077
...,...
5175,2.433613
686,2.785011
4631,1.492904
5701,4.110874


In [64]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import KFold, cross_validate

def Multiple_Ml_Objective(trial):

    # 1. Select the Model
    regressor_name = trial.suggest_categorical('regressor', ['Random Forest', 'Extra Trees', 'Xgboost'])

    # 2. Random Forest Branch
    if regressor_name == 'Random Forest':
        n_estimators = trial.suggest_int('rf_n_estimators', 100, 800, step=100) 
        criterion = trial.suggest_categorical('rf_criterion', ['squared_error', 'absolute_error'])
        
        max_depth_option = trial.suggest_categorical('rf_max_depth_option', ['auto', 'fixed'])
        max_depth = None if max_depth_option == 'auto' else trial.suggest_int('rf_max_depth', 10, 150, step=10)
        
        min_samples_split = trial.suggest_int('rf_min_samples_split', 2, 30)
        min_samples_leaf = trial.suggest_int('rf_min_samples_leaf', 1, 20)
        max_features = trial.suggest_categorical('rf_max_features', ['sqrt', 'log2', 1.0, 0.3, 0.5, 0.75])
        max_samples = trial.suggest_float('rf_max_samples', 0.4, 0.95)
        
        # NEW: Tree Pruning & Size Control
        max_leaf_nodes = trial.suggest_int('rf_max_leaf_nodes', 50, 2000, log=True)
        ccp_alpha = trial.suggest_float('rf_ccp_alpha', 0.0, 0.1)

        model = RandomForestRegressor(
            n_estimators=n_estimators, 
            criterion=criterion, 
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features, 
            max_samples=max_samples,
            max_leaf_nodes=max_leaf_nodes,
            ccp_alpha=ccp_alpha,
            n_jobs=-1,
            random_state=42
        )

    # 3. Extra Trees Branch
    elif regressor_name == 'Extra Trees':
        n_estimators = trial.suggest_int('et_n_estimators', 100, 800, step=100)
        criterion = trial.suggest_categorical('et_criterion', ['squared_error', 'absolute_error'])
        
        max_depth_option = trial.suggest_categorical('et_max_depth_option', ['auto', 'fixed'])
        max_depth = None if max_depth_option == 'auto' else trial.suggest_int('et_max_depth', 10, 150, step=10)
        
        min_samples_split = trial.suggest_int('et_min_samples_split', 2, 30)
        min_samples_leaf = trial.suggest_int('et_min_samples_leaf', 1, 20)
        max_features = trial.suggest_categorical('et_max_features', ['sqrt', 'log2', 1.0, 0.3, 0.5, 0.75])
        
        # NEW: Bootstrapping & Pruning 
        bootstrap = trial.suggest_categorical('et_bootstrap', [True, False])
        max_samples = trial.suggest_float('et_max_samples', 0.4, 0.95) if bootstrap else None
        max_leaf_nodes = trial.suggest_int('et_max_leaf_nodes', 50, 2000, log=True)
        ccp_alpha = trial.suggest_float('et_ccp_alpha', 0.0, 0.1)

        model = ExtraTreesRegressor(
            n_estimators=n_estimators, 
            criterion=criterion, 
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features, 
            bootstrap=bootstrap,
            max_samples=max_samples,
            max_leaf_nodes=max_leaf_nodes,
            ccp_alpha=ccp_alpha,
            n_jobs=-1,
            random_state=42
        )

    # 4. XGBoost Branch
    elif regressor_name == 'Xgboost':
        n_estimators = trial.suggest_int('xgb_n_estimators', 200, 1000, step=100)
        max_depth = trial.suggest_int('xgb_max_depth', 3, 20)
        learning_rate = trial.suggest_float('xgb_learning_rate', 1e-4, 0.3, log=True)
        subsample = trial.suggest_float('xgb_subsample', 0.5, 1.0)
        colsample_bytree = trial.suggest_float('xgb_colsample_bytree', 0.4, 1.0)
        gamma = trial.suggest_float('xgb_gamma', 0.0, 10.0)
        
        # NEW: L1/L2 Regularization and Node Splitting Control
        min_child_weight = trial.suggest_int('xgb_min_child_weight', 1, 20)
        reg_alpha = trial.suggest_float('xgb_reg_alpha', 1e-5, 10.0, log=True) # L1
        reg_lambda = trial.suggest_float('xgb_reg_lambda', 1e-5, 10.0, log=True) # L2
        colsample_bylevel = trial.suggest_float('xgb_colsample_bylevel', 0.4, 1.0)

        model = XGBRegressor(
            n_estimators=n_estimators,
            max_depth=max_depth,
            learning_rate=learning_rate,
            subsample=subsample,
            colsample_bytree=colsample_bytree,
            colsample_bylevel=colsample_bylevel,
            gamma=gamma,
            min_child_weight=min_child_weight,
            reg_alpha=reg_alpha,
            reg_lambda=reg_lambda,
            n_jobs=-1,
            random_state=42
        )

    # 5. Cross Validation
    cv_strategy = KFold(n_splits=5, shuffle=True, random_state=42)

    cv_results = cross_validate(
        model,
        X_train_transformed,
        y_train['Price(Cr)'], # Note: Ensure you use y_train_log here if you want to keep optimizing in log-space!
        scoring='r2',
        cv=cv_strategy,
        return_train_score=True,
        n_jobs=-1 # Speeds up cross-validation
    )

    # 6. Extract Metrics
    train_score_mean = cv_results['train_score'].mean()
    val_score_mean = cv_results['test_score'].mean()

    # 7. Log Custom Attributes to Optuna Dashboard
    trial.set_user_attr('train_score_mean', train_score_mean)
    trial.set_user_attr('train_score_std', cv_results['train_score'].std())
    trial.set_user_attr('test_score_std', cv_results['test_score'].std())
    trial.set_user_attr('overfitting_gap', train_score_mean - val_score_mean)

    return (val_score_mean)

In [65]:
multiple_ml_study = optuna.create_study(direction='maximize',sampler=optuna.samplers.TPESampler())
multiple_ml_study.optimize(Multiple_Ml_Objective,n_trials=5)

[I 2026-05-16 12:43:43,750] A new study created in memory with name: no-name-ff71c7f1-dfed-47d5-aea8-4fb8c414abf1
[I 2026-05-16 12:43:52,204] Trial 0 finished with value: 0.6563412448496818 and parameters: {'regressor': 'Extra Trees', 'et_n_estimators': 600, 'et_criterion': 'squared_error', 'et_max_depth_option': 'auto', 'et_min_samples_split': 28, 'et_min_samples_leaf': 17, 'et_max_features': 'log2', 'et_bootstrap': True, 'et_max_samples': 0.6686775179561499, 'et_max_leaf_nodes': 1030, 'et_ccp_alpha': 0.0032586832300847248}. Best is trial 0 with value: 0.6563412448496818.
[I 2026-05-16 12:43:54,655] Trial 1 finished with value: 0.9381345526646614 and parameters: {'regressor': 'Xgboost', 'xgb_n_estimators': 800, 'xgb_max_depth': 5, 'xgb_learning_rate': 0.013111863963010647, 'xgb_subsample': 0.7209642155583973, 'xgb_colsample_bytree': 0.7984155383632925, 'xgb_gamma': 3.176398934879666, 'xgb_min_child_weight': 7, 'xgb_reg_alpha': 0.06183341526055012, 'xgb_reg_lambda': 0.18360594514459488

In [66]:
df = multiple_ml_study.trials_dataframe()

In [67]:
df.head()

,number,value,datetime_start,datetime_complete,duration,params_et_bootstrap,params_et_ccp_alpha,params_et_criterion,params_et_max_depth_option,params_et_max_features,...,params_xgb_min_child_weight,params_xgb_n_estimators,params_xgb_reg_alpha,params_xgb_reg_lambda,params_xgb_subsample,user_attrs_overfitting_gap,user_attrs_test_score_std,user_attrs_train_score_mean,user_attrs_train_score_std,state
0,0,0.656341,2026-05-16 12:43:43.752842,2026-05-16 12:43:52.204652,0 days 00:00:08.451810,True,0.003259,squared_error,auto,log2,...,NaN,NaN,NaN,NaN,NaN,0.004170,0.032344,0.660511,0.003293,COMPLETE
1,1,0.938135,2026-05-16 12:43:52.207245,2026-05-16 12:43:54.654990,0 days 00:00:02.447745,NaN,NaN,NaN,NaN,NaN,...,7.0,800.0,0.061833,0.183606,0.720964,0.033709,0.006713,0.971843,0.001092,COMPLETE
2,2,0.887859,2026-05-16 12:43:54.656776,2026-05-16 12:44:42.331426,0 days 00:00:47.674650,True,0.070295,absolute_error,auto,0.75,...,NaN,NaN,NaN,NaN,NaN,0.006446,0.015463,0.894306,0.004187,COMPLETE
3,3,0.926119,2026-05-16 12:44:42.332403,2026-05-16 12:44:44.053156,0 days 00:00:01.720753,NaN,NaN,NaN,NaN,NaN,...,4.0,300.0,0.006015,0.003692,0.668470,0.038512,0.009066,0.964631,0.000927,COMPLETE
4,4,0.862093,2026-05-16 12:44:44.059666,2026-05-16 12:44:53.277762,0 days 00:00:09.218096,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.015171,0.009693,0.877263,0.002915,COMPLETE


In [68]:
df.groupby('params_regressor')['value'].mean()

params_regressor
Extra Trees      0.772100
Random Forest    0.862093
Xgboost          0.932127
Name: value, dtype: float64

In [69]:
df.groupby('params_regressor')['user_attrs_overfitting_gap'].mean()

params_regressor
Extra Trees      0.005308
Random Forest    0.015171
Xgboost          0.036110
Name: user_attrs_overfitting_gap, dtype: float64

In [70]:
df.sort_values(by=['value','user_attrs_overfitting_gap'],ascending=[True,True]).head()

,number,value,datetime_start,datetime_complete,duration,params_et_bootstrap,params_et_ccp_alpha,params_et_criterion,params_et_max_depth_option,params_et_max_features,...,params_xgb_min_child_weight,params_xgb_n_estimators,params_xgb_reg_alpha,params_xgb_reg_lambda,params_xgb_subsample,user_attrs_overfitting_gap,user_attrs_test_score_std,user_attrs_train_score_mean,user_attrs_train_score_std,state
0,0,0.656341,2026-05-16 12:43:43.752842,2026-05-16 12:43:52.204652,0 days 00:00:08.451810,True,0.003259,squared_error,auto,log2,...,NaN,NaN,NaN,NaN,NaN,0.004170,0.032344,0.660511,0.003293,COMPLETE
4,4,0.862093,2026-05-16 12:44:44.059666,2026-05-16 12:44:53.277762,0 days 00:00:09.218096,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.015171,0.009693,0.877263,0.002915,COMPLETE
2,2,0.887859,2026-05-16 12:43:54.656776,2026-05-16 12:44:42.331426,0 days 00:00:47.674650,True,0.070295,absolute_error,auto,0.75,...,NaN,NaN,NaN,NaN,NaN,0.006446,0.015463,0.894306,0.004187,COMPLETE
3,3,0.926119,2026-05-16 12:44:42.332403,2026-05-16 12:44:44.053156,0 days 00:00:01.720753,NaN,NaN,NaN,NaN,NaN,...,4.0,300.0,0.006015,0.003692,0.668470,0.038512,0.009066,0.964631,0.000927,COMPLETE
1,1,0.938135,2026-05-16 12:43:52.207245,2026-05-16 12:43:54.654990,0 days 00:00:02.447745,NaN,NaN,NaN,NaN,NaN,...,7.0,800.0,0.061833,0.183606,0.720964,0.033709,0.006713,0.971843,0.001092,COMPLETE


In [57]:
# Extract the best overall score (lowest MAE)
print("Best Validation MAE (in log-space):", multiple_ml_study.best_value)

# Extract the parameters that achieved this score
best_params = multiple_ml_study.best_params
print("\nBest Hyperparameters Found:")
for key, value in best_params.items():
    print(f"  {key}: {value}")

# Extract the custom attributes we logged for that specific trial
best_trial_attrs = multiple_ml_study.best_trial.user_attrs
print("\nBest Trial Diagnostics:")
print(f"  Training Score Mean: {best_trial_attrs['train_score_mean']}")
print(f"  Overfitting Gap: {best_trial_attrs['overfitting_gap']}")

Best Validation MAE (in log-space): 0.15121231844947855

Best Hyperparameters Found:
  regressor: Xgboost
  xgb_n_estimators: 1000
  xgb_max_depth: 4
  xgb_learning_rate: 0.014248835283463022
  xgb_subsample: 0.7183593504505594
  xgb_colsample_bytree: 0.6699139158385192
  xgb_gamma: 2.514185493912456
  xgb_min_child_weight: 13
  xgb_reg_alpha: 0.13107784039877307
  xgb_reg_lambda: 0.0010238445051984043
  xgb_colsample_bylevel: 0.5915922772734974

Best Trial Diagnostics:
  Training Score Mean: -0.14604170729642768
  Overfitting Gap: 0.005170611153050869


In [62]:
# 1. Figure out which model type won
winning_model_type = best_params['regressor']
print(f"The winning architecture is: {winning_model_type}")

# 2. Filter out the parameters belonging only to the winning model
# and strip the prefix (e.g., 'rf_n_estimators' becomes 'n_estimators')
clean_params = {}
prefix = ""
if winning_model_type == 'Random Forest':
    prefix = 'rf_'
elif winning_model_type == 'Extra Trees':
    prefix = 'et_'
elif winning_model_type == 'Xgboost':
    prefix = 'xgb_'

for key, value in best_params.items():
    if key.startswith(prefix):
        clean_key = key.replace(prefix, '')
        clean_params[clean_key] = value

# Handle the max_depth_option conditional mapping we created for sklearn models
if 'max_depth_option' in clean_params:
    if clean_params['max_depth_option'] == 'auto':
        clean_params['max_depth'] = None
    del clean_params['max_depth_option']

# 3. Instantiate the winning model type with the optimized parameters
if winning_model_type == 'Random Forest':
    final_model = RandomForestRegressor(**clean_params, n_jobs=-1, random_state=42)
elif winning_model_type == 'Extra Trees':
    final_model = ExtraTreesRegressor(**clean_params, n_jobs=-1, random_state=42)
elif winning_model_type == 'Xgboost':
    final_model = XGBRegressor(**clean_params, n_jobs=-1, random_state=42)

# 4. Train the final model on your ENTIRE training dataset
final_model.fit(X_train_transformed, y_train['Price(Cr)'])
print("Final model is fully trained and ready for serialization!")

The winning architecture is: Xgboost
Final model is fully trained and ready for serialization!


In [63]:
y_pred = final_model.predict(X_test_transformed)
# y_pred = np.expm1(y_pred)
print('r2 Score',r2_score(y_test,y_pred))
print('Mean absolute error',mean_absolute_error(y_test,y_pred))

r2 Score 0.9406909346580505
Mean absolute error 1.421180248260498


In [71]:
# 1. Figure out which model type won
winning_model_type = best_params['regressor']
print(f"The winning architecture is: {winning_model_type}")

# 2. Filter out the parameters belonging only to the winning model
# and strip the prefix (e.g., 'rf_n_estimators' becomes 'n_estimators')
clean_params = {}
prefix = ""
if winning_model_type == 'Random Forest':
    prefix = 'rf_'
elif winning_model_type == 'Extra Trees':
    prefix = 'et_'
elif winning_model_type == 'Xgboost':
    prefix = 'xgb_'

for key, value in best_params.items():
    if key.startswith(prefix):
        clean_key = key.replace(prefix, '')
        clean_params[clean_key] = value

# Handle the max_depth_option conditional mapping we created for sklearn models
if 'max_depth_option' in clean_params:
    if clean_params['max_depth_option'] == 'auto':
        clean_params['max_depth'] = None
    del clean_params['max_depth_option']

# 3. Instantiate the winning model type with the optimized parameters
if winning_model_type == 'Random Forest':
    final_model = RandomForestRegressor(**clean_params, n_jobs=-1, random_state=42)
elif winning_model_type == 'Extra Trees':
    final_model = ExtraTreesRegressor(**clean_params, n_jobs=-1, random_state=42)
elif winning_model_type == 'Xgboost':
    final_model = XGBRegressor(**clean_params, n_jobs=-1, random_state=42)

# 4. Train the final model on your ENTIRE training dataset
final_model.fit(X_train_transformed, y_train['Price(Cr)'])
print("Final model is fully trained and ready for serialization!")

The winning architecture is: Xgboost
Final model is fully trained and ready for serialization!


In [72]:
y_pred = final_model.predict(X_test_transformed)
# y_pred = np.expm1(y_pred)
print('r2 Score',r2_score(y_test,y_pred))
print('Mean absolute error',mean_absolute_error(y_test,y_pred))

r2 Score 0.9406909346580505
Mean absolute error 1.421180248260498
